# Descarga y almacenamiento de datos de jugadores de ligas europeas

Este código permite descargar información de jugadores de las principales ligas europeas (Premier League, La Liga, Serie A, Bundesliga y Ligue 1) usando la API de **Understat** (https://understat.com/). Los datos incluyen estadísticas detalladas de cada jugador por temporada.

Se utilizan las siguientes librerías principales:
- **asyncio** y **aiohttp**: Para realizar solicitudes HTTP de manera asíncrona y eficiente.
    - **understat**: Biblioteca para acceder a la API de Understat.
- **pandas**: Para manipulación y almacenamiento de los datos en formato tabular.
- **os**: Para la gestión de rutas y creación de carpetas.
- **nest_asyncio**: Para permitir la ejecución de código asíncrono en notebooks de Jupyter.

El flujo de trabajo es el siguiente:
1. Se recorren varias temporadas y ligas.
2. Para cada liga y temporada, se descarga la información de los jugadores mediante llamadas asíncronas a Understat.
3. Cada conjunto de datos se guarda como un CSV individual en la ruta:
4. Además, se genera un CSV combinado por temporada que incluye todos los jugadores de todas las ligas

datos/<NombreLiga>/<Temporada>/players.csv 

datos/combined/all_leagues_<Temporada>.csv  ->
Esto permite tener tanto archivos individuales por liga como un dataset maestro por temporada listo para análisis y procesamiento posterior.


📊 Descripción de cada columna

- 🆔 Identificación

id → Identificador único del jugador en Understat.

player_name → Nombre del jugador.

- 🕒 Participación

games → Partidos jugados.

time → Minutos totales jugados.

- ⚽ Rendimiento ofensivo

goals → Goles anotados (totales).

assists → Asistencias registradas oficialmente.

shots → Tiros realizados.

- 📈 Estadísticas avanzadas (xG / xA)

xG (Expected Goals)
Probabilidad estimada de que un jugador convierta sus tiros en gol según calidad de las ocasiones.

xA (Expected Assists)
Probabilidad de que un pase termine en tiro peligroso, mide la calidad de las oportunidades creadas.

npg (Non-Penalty Goals)
Goles anotados excluyendo penales.

npxG (Non-Penalty Expected Goals)
xG excluyendo tiros de penal (más preciso para medir finalización real).

- 🎯 Creación de juego y contribución en ataque

key_passes → Pases que generan un tiro (no necesariamente gol).

xGChain
xG acumulado de todas las posesiones en las que el jugador participó (tocó la pelota en la jugada), refleja contribución total al ataque, incluso si no tiró o dio asistencia.

xGBuildup
Similar a xGChain, pero excluye tiros y pases clave. Mide participación en la construcción del ataque (toques previos).

- 🟨 Disciplina

yellow_cards → Tarjetas amarillas.

red_cards → Tarjetas rojas.

- 🧩 Información contextual

position → Posición del jugador (DEF, MID, FWD, etc.).

team_title → Nombre del equipo.

In [48]:
import asyncio
from understat import Understat
import pandas as pd
import os
from aiohttp import ClientSession
from tqdm.notebook import tqdm

# Diccionarios de las ligas y el año inicial por temporada
leagues = {
    "EPL": "Premier-League",
    "La_liga": "La-Liga",
    "Serie_A": "Serie-A",
    "Bundesliga": "Bundesliga",
    "Ligue_1": "Ligue-1"
}

seasons = ["2020", "2021", "2022", "2023", "2024"]

BASE_DIR = "datos"
os.makedirs(BASE_DIR, exist_ok=True)

# Función asíncrona para obtener jugadores de una liga y temporada
async def get_players(league_code, season):
    async with ClientSession() as session:
        understat = Understat(session)
        players = await understat.get_league_players(league_code, season)
        df = pd.DataFrame(players)
        return df

# Loop principal para descargar y guardar
async def main():
    all_seasons_combined = {}

    for season in seasons:
        season_combined = pd.DataFrame()
        for code, league_name in leagues.items():
            print(f"\n📄 Descargando {league_name} temporada {season}-{int(season)+1}")
            try:
                df = await get_players(code, season)
            except Exception as e:
                print(f"⚠️ Error descargando {league_name} {season}-{int(season)+1}: {e}")
                continue

            # Guardar CSV por liga y temporada
            out_dir = os.path.join(BASE_DIR, league_name, f"{season}-{int(season)+1}")
            os.makedirs(out_dir, exist_ok=True)
            df.to_csv(os.path.join(out_dir, "players.csv"), index=False)
            print(f"✅ Guardado {league_name} {season}-{int(season)+1} players.csv")

            # Añadir columna de liga y combinar
            df["league"] = league_name
            season_combined = pd.concat([season_combined, df], ignore_index=True)

        # Guardar CSV maestro de la temporada
        all_seasons_combined[season] = season_combined
        combined_dir = os.path.join(BASE_DIR, "combined")
        os.makedirs(combined_dir, exist_ok=True)
        season_combined.to_csv(os.path.join(combined_dir, f"all_leagues_{season}-{int(season)+1}.csv"), index=False)
        print(f"💾 Guardado CSV combinado para la temporada {season}-{int(season)+1}")

# Ejecutar
import nest_asyncio
nest_asyncio.apply()
await main()



📄 Descargando Premier-League temporada 2020-2021
⚠️ Error descargando Premier-League 2020-2021: expected string or bytes-like object

📄 Descargando La-Liga temporada 2020-2021
⚠️ Error descargando La-Liga 2020-2021: expected string or bytes-like object

📄 Descargando Serie-A temporada 2020-2021
⚠️ Error descargando Serie-A 2020-2021: expected string or bytes-like object

📄 Descargando Bundesliga temporada 2020-2021
⚠️ Error descargando Bundesliga 2020-2021: expected string or bytes-like object

📄 Descargando Ligue-1 temporada 2020-2021
⚠️ Error descargando Ligue-1 2020-2021: expected string or bytes-like object
💾 Guardado CSV combinado para la temporada 2020-2021

📄 Descargando Premier-League temporada 2021-2022
⚠️ Error descargando Premier-League 2021-2022: expected string or bytes-like object

📄 Descargando La-Liga temporada 2021-2022
⚠️ Error descargando La-Liga 2021-2022: expected string or bytes-like object

📄 Descargando Serie-A temporada 2021-2022
⚠️ Error descargando Serie-A 2

# Descarga de datos por jornada

Este código permite descargar información actualizada de los jugadores de **La Liga** jornada a jornada utilizando la API de **Understat** (https://understat.com/). Está pensado para seguir la evolución de las estadísticas de los jugadores durante toda la temporada.

Se utilizan las siguientes librerías principales:
- **asyncio** y **aiohttp**: Para realizar las solicitudes HTTP de forma asíncrona y eficiente.
- **understat**: Biblioteca para acceder a los datos de Understat.
- **pandas**: Para organizar y guardar los datos en formato tabular (CSV).
- **os**: Para la creación automática de carpetas y gestión de rutas.
- **datetime**: Para registrar la fecha y hora exacta de cada descarga.
- **nest_asyncio**: Para permitir la ejecución de código asíncrono dentro de notebooks de Jupyter.

El flujo de trabajo es el siguiente:
1. Se conecta a la API de Understat y descarga los datos más recientes de los jugadores de La Liga.
2. Se especifica la jornada actual o se detecta automáticamente la siguiente jornada disponible.
3. Se guarda la información en un archivo CSV dentro de una carpeta correspondiente a la jornada.
4. Cada descarga queda registrada con la fecha y hora en la que se realizó, lo que permite llevar un seguimiento temporal de la evolución de los jugadores.

La estructura de carpetas generada es la siguiente:

```
datos/La-Liga/<Temporada>/Jornada_<número>/players_jornada_<número>.csv
```

Por ejemplo:
```
datos/La-Liga/2025-2026/Jornada_10/players_jornada_10.csv
```

Además, el script puede detectar automáticamente la **siguiente jornada** pendiente de descarga mediante la función `descargar_siguiente_jornada()`, lo que facilita la actualización semanal de los datos sin intervención manual.

Este sistema permite construir un histórico de rendimiento por jornada, ideal para análisis de evolución de jugadores, rendimiento acumulado o comparativas entre jornadas.


In [49]:
import asyncio
from understat import Understat
import pandas as pd
import os
from aiohttp import ClientSession
from datetime import datetime
import nest_asyncio

CURRENT_SEASON = "2025"
LEAGUE_NAME = "La-Liga"

nest_asyncio.apply()

async def datos_jornada(jornada):
    async with ClientSession() as session:
        understat = Understat(session)
        league_code = "La_liga"

        season_dir = os.path.join("datos", LEAGUE_NAME, f"{CURRENT_SEASON}-{int(CURRENT_SEASON)+1}")
        jornada_dir = os.path.join(season_dir, f"Jornada_{jornada}")
        os.makedirs(jornada_dir, exist_ok=True)

        print(f"\n⚽ Descargando datos de {LEAGUE_NAME} jornada {jornada} ({CURRENT_SEASON}-{int(CURRENT_SEASON)+1})...")

        try:
            players_raw = await understat.get_league_players(league_code, CURRENT_SEASON)

            # Algunos dicts vienen anidados, extraemos solo la lista de jugadores
            if isinstance(players_raw, dict) and "players" in players_raw:
                players = players_raw["players"]
            else:
                players = players_raw

            df = pd.DataFrame(players)

            df["league"] = LEAGUE_NAME
            df["season"] = f"{CURRENT_SEASON}-{int(CURRENT_SEASON)+1}"
            df["jornada"] = jornada
            df["fecha_descarga"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            csv_path = os.path.join(jornada_dir, f"players_jornada_{jornada}.csv")
            df.to_csv(csv_path, index=False)

            print(f"✅ Guardado {csv_path}")
            return df

        except Exception as e:
            print(f"⚠️ Error al descargar los datos de {LEAGUE_NAME} jornada {jornada}: {e}")
            return None

# Función para detectar la siguiente jornada
async def descargar_siguiente_jornada():
    season_dir = os.path.join("datos", LEAGUE_NAME, f"{CURRENT_SEASON}-{int(CURRENT_SEASON)+1}")
    os.makedirs(season_dir, exist_ok=True)

    jornadas_existentes = [
        int(folder.replace("Jornada_", "")) 
        for folder in os.listdir(season_dir) 
        if folder.startswith("Jornada_")
    ]

    siguiente_jornada = max(jornadas_existentes) + 1 if jornadas_existentes else 1
    print(f"📅 Última jornada detectada: {jornadas_existentes[-1] if jornadas_existentes else 'Ninguna'}")
    print(f"🚀 Descargando Jornada {siguiente_jornada}...")

    await datos_jornada(siguiente_jornada)

# Ejecutamos
await descargar_siguiente_jornada()


📅 Última jornada detectada: 14
🚀 Descargando Jornada 15...

⚽ Descargando datos de La-Liga jornada 15 (2025-2026)...
⚠️ Error al descargar los datos de La-Liga jornada 15: expected string or bytes-like object


In [10]:
import nest_asyncio
nest_asyncio.apply()

In [15]:
# Jornada 10
await datos_jornada(10)


⚽ Descargando datos de La-Liga jornada 10 (2025-2026)...
✅ Guardado datos\La-Liga\2025-2026\Jornada_10\players_jornada_10.csv


,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,...,position,team_title,npg,npxG,xGChain,xGBuildup,league,season,jornada,fecha_descarga
0,3423,Kylian Mbappe-Lottin,10,886,11,9.70220810174942,2,2.5150726065039635,52,27,...,F,Real Madrid,9,7.472375184297562,10.809774279594421,2.7999667823314667,La-Liga,2025-2026,10,2025-10-30 16:00:57
1,10846,Julián Álvarez,10,794,6,5.159378491342068,2,3.0665366761386395,22,17,...,F,Atletico Madrid,5,3.6728232875466347,7.637867271900177,3.0547057799994946,La-Liga,2025-2026,10,2025-10-30 16:00:57
2,10930,Etta Eyong,10,826,6,6.505741871893406,3,1.6703463960438967,24,8,...,F,"Levante,Villarreal",6,6.505741871893406,6.4708163514733315,0.3054979722946882,La-Liga,2025-2026,10,2025-10-30 16:00:57
3,7008,Vinícius Júnior,10,716,5,3.756085805594921,4,2.9066833779215813,27,22,...,F M S,Real Madrid,4,3.012808196246624,7.838781118392944,2.2023639008402824,La-Liga,2025-2026,10,2025-10-30 16:00:57
4,9002,Vedat Muriqi,9,735,5,3.560269933193922,0,0.12538825068622828,25,4,...,F,Mallorca,4,2.816992323845625,4.052325740456581,1.7577132657170296,La-Liga,2025-2026,10,2025-10-30 16:00:57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,14018,Lucas Ahijado,5,367,0,0.3388008251786232,1,0.8013708367943764,3,3,...,D S,Real Oviedo,0,0.3388008251786232,1.3727248758077621,0.9944158587604761,La-Liga,2025-2026,10,2025-10-30 16:00:57
473,14050,Kervin Arriaga,7,374,0,0.03903232142329216,0,0.10146134346723557,2,2,...,M S,Levante,0,0.03903232142329216,1.1069653555750847,0.9971281215548515,La-Liga,2025-2026,10,2025-10-30 16:00:57
474,14064,Roony Bardghji,6,156,0,0.9467628411948681,0,0.8982919603586197,9,5,...,M S,Barcelona,0,0.9467628411948681,2.318600073456764,1.664001453667879,La-Liga,2025-2026,10,2025-10-30 16:00:57
475,14086,Alex Sancris,5,267,0,0.05582698993384838,0,0.24741654843091965,3,3,...,F S,Getafe,0,0.05582698993384838,0.33476503752171993,0.22421850264072418,La-Liga,2025-2026,10,2025-10-30 16:00:57
